In [ ]:
%run Setting_Env_Variables_p2.py
google_project_id = %env GOOGLE_CLOUD_PROJECT
bucket = os.getenv("WORKSPACE_BUCKET")

import os
import subprocess
import pandas as pd
from datetime import datetime

import hail as hl
hl.init(gcs_requester_pays_configuration=google_project_id) #, log=f'{bucket}/hail_logs')
hl.default_reference(new_default_reference = "GRCh38")


In [ ]:
auxiliary_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux"
print(f'aux path: {auxiliary_path}')

vat_path = f'{auxiliary_path}/vat/*.gz'
print(f'vat path: {vat_path}')

clinvar_ht_path = 'gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/clinvar/multiMT/hail.mt'
print(f'clinvar ht path: {clinvar_ht_path}')

vds_path = 'gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/vds/hail.vds'
print(f'vds path: {vds_path}')

flagged_samples_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/qc/flagged_samples.tsv"
print(f'flagged samples path: {flagged_samples_path}')

exons = f"{bucket}/exons_chr1_155082572_157140081.bed"
exons_interval_table = hl.import_bed(exons, reference_genome='GRCh38')
# introns_interval_table = hl.import_locus_intervals('/home/dataproc/repos/variant-effects/gene_info/introns_chr1_155082572_157140081.bed', reference_genome='GRCh38')

# mt_wgs_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/multiMT/hail.mt"
# flagged_samples = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/qc/flagged_samples.tsv"
# array_path = "gs://vwb-aou-datasets-controlled/v9/microarray/hail.mt"
# # this path is in the official ct v8 fw
# bed_path = "gs://aou-tutorial-notebooks-wb-sunny-radish-6214/genomic_test_data/random_sites_GRCh38.txt"

In [ ]:
%%time

types={'clinvar_classification': hl.tarray(hl.tstr), 
       'position': hl.tint, 
       'omim_phenotypes_id':hl.tarray(hl.tint), 
       'omim_phenotypes_name':hl.tarray(hl.tstr),
       'clinvar_phenotype':hl.tarray(hl.tstr)
      }



vat_table = hl.import_table(vat_path, force=True, quote='"', delimiter="\t", force_bgz=True, types=types) # impute=True)
vat_table = vat_table.add_index(name='id')

parse by demographic

Get all LMNA associated variants for control groups

In [ ]:
gene = 'LMNA'
ens_id = 'ENSG00000160789'

lmna_vat = vat_table.filter(vat_table["gene_id"]==ens_id)

In [ ]:
lmna_vat.describe()

In [ ]:
# filt_vat.write(f'{bucket}/hail_files/lmna_vat.ht', overwrite=True)
# filt_vat.describe()

In [ ]:
filtered_ht = filt_vat.filter(hl.any(
    # filt_vat.consequence.contains('intron_variant'),
    # filt_vat.consequence.contains('non_coding_transcript_variant'),
    filt_vat.clinvar_classification.contains('pathogenic'),
    filt_vat.clinvar_classification.contains('uncertain significance'),
    filt_vat.clinvar_classification.contains('likely pathogenic'),
))

In [ ]:
# filtered_ht.write(f'{bucket}/hail_files/lmna_vat.ht', overwrite=True)


In [ ]:
# print(f'{gene} VAT length = {filtered_ht.count()}')

# get variants within 1 Mb of LMNA

In [ ]:
ct = hl.read_matrix_table(clinvar_ht_path)
# ct.describe()


%%time
ct.count()

# vds

In [ ]:
%%time
vds = hl.vds.read_vds(vds_path)

In [ ]:
# only work with data from chromosome 1 (location of LMNA)
vds = hl.vds.filter_chromosomes(vds, keep= ["chr1"])

In [ ]:
# change coordinates to 1Mb on either side of LMNA
#chr1:156082572-156140081
start = 156082572
stop = 156140081
interval = f'chr1:{start-1000000}-{stop+1000000}'
intervals = [interval]
print(interval)


In [ ]:
%%time
# get only variants within the defined interval above
vds = hl.vds.filter_intervals(
    vds,
    [hl.parse_locus_interval(x,)
     for x in intervals])

In [ ]:
# %%time
# vds.variant_data.count() (within interval)

# count() output:

In [ ]:
%%time
# get only variants that are not in exons within 1 Mb of LMNA
vds = hl.vds.filter_intervals(
    vds,
    exons_interval_table.interval.collect(),
    keep=False)

In [ ]:
# vds.variant_data.count()

# 1 Mb before LMNA (start-1Mb, start): (384498, 535662)

The field sample_id contains the person_ids, so we set the field sample_id as key while reading the table.

In [ ]:
flagged_samples = hl.import_table(flagged_samples_path, key = "s")

In [ ]:
# filter out flagged samples
vds = hl.vds.filter_samples(vds, flagged_samples, keep = False, remove_dead_alleles = True)

In [ ]:
# %%time
# vds.variant_data.count()

# 1 Mb before LMNA (start-1Mb, start): (382732, 534650) 7min 29s

In [ ]:
vds.variant_data.describe()

Transform Local Allelic Depth (LAD)
The All of Us VDS uses Local Alleles (LA) which has information for reference alleles and all alternative alleles across all samples in the array. We will need to convert Local Alleles related fields, Local Allele Depth (LAD) and Local Genotypte (LGT) to the Global Allele format fields (AD and GT), which have information for the reference allele and alternative allele for each sample in the array. We will use two diferent functions to convert LAD to AD and LGT to GT.



In [ ]:
# Convert LAD to AD using the function annotate_entries and vds.local_to_global
mt = vds.variant_data.annotate_entries(AD = hl.vds.local_to_global(vds.variant_data.LAD, 
                                                                   vds.variant_data.LA, 
                                                                   n_alleles=hl.len(vds.variant_data.alleles), 
                                                                   fill_value=0, number='R'))

In [ ]:
# Convert LGT to GT using the function lgt_to_gt
mt = mt.annotate_entries(GT = hl.vds.lgt_to_gt(mt.LGT, mt.LA))

In [ ]:
# Tranform FT field
# The field FT is boolean, which is not accepted by VCF. We will tranform it to "Fail" or "PASS", using the function transmute_entry.
mt = mt.transmute_entries(FT = hl.if_else(mt.FT, "PASS", "FAIL"))

In [ ]:
# densify the matrixtable
mt = hl.vds.to_dense_mt(hl.vds.VariantDataset(vds.reference_data, mt))

In [ ]:
# Create fields AC / AF / AN
# There are no Allele Counts (AC) / Allele Frequency (AF) / Allele Number (AN) in the VDS. We will use the function agg.call_stats to get the AC / AF / AN.
mt = mt.annotate_rows(info = hl.agg.call_stats(mt.GT, mt.alleles))

In [ ]:
mt = mt.annotate_entries(GT=hl.vds.lgt_to_gt(mt.LGT, mt.LA))

In [ ]:
# # drop unnecessary fields
# fields_to_drop_list = ['as_vets','as_vqsr','LAD', 'LGT', 'LA',
#             'tranche_data', 'truth_sensitivity_snp_threshold', 
#              'truth_sensitivity_indel_threshold','snp_vqslod_threshold','indel_vqslod_threshold']

# mt = mt.drop(*(f for f in fields_to_drop_list if f in mt.entry or f in mt.row or f in mt.col or f in mt.globals))
mt.describe()

In [ ]:
# sc = hl.spark_context()
# print(sc.uiWebUrl)

In [ ]:
# Only use if a single node is choking on data skew
mt = mt.repartition(500, shuffle=True)


In [ ]:
%%time
out_path = f'{bucket}/hail_files/variants_notExons_1MbLMNA_pre.mt'
mt.write(out_path, overwrite = True)

In [ ]:
# out_path = f'{bucket}/hail_files/variants_notExons_1MbLMNA.mt'
# mt.write(out_path, overwrite = True)




In [ ]:
# vds.variant_data.count()